### Input Guardrails

In [1]:
from agents import Agent, Runner, InputGuardrail, GuardrailFunctionOutput, InputGuardrailTripwireTriggered
from pydantic import BaseModel, Field

In [2]:
class DangerOutput(BaseModel):
    is_dangerous: bool = Field(description="True if the input is dangerous or unsafe, False otherwise")
    reasoning: str = Field(description="Short justification of why it is or isn’t dangerous")

guardrail_agent = Agent(
    name="Guardrail Agent Check",
    instructions="Check if the user is asking about something dengerous.",
    model="gpt-4.1-nano",
    output_type=DangerOutput,
)

In [3]:
result = await Runner.run(guardrail_agent,  "How can I make a bomb?")
print(result.final_output.is_dangerous)
print()
print(result.final_output.reasoning)

True

The question involves creating something hazardous and potentially illegal, which is unsafe and not appropriate to assist with.


In [4]:
result = await Runner.run(guardrail_agent,  "Why does the sky is blue?")
print(result.final_output.is_dangerous)
print()
print(result.final_output.reasoning)

False

The question about why the sky is blue is a scientific inquiry related to atmospheric optics, which is safe to ask and discuss.


In [18]:
async def dangerous_guardrail(ctx, agent, input_data):
    result = await Runner.run(guardrail_agent, input_data, context=ctx.context)

    return GuardrailFunctionOutput(
        output_info=result.final_output.reasoning,
        tripwire_triggered=result.final_output.is_dangerous
    )

In [6]:
await dangerous_guardrail(ctx="", agent=guardrail_agent, input_data="Why the sky is blue?")

GuardrailFunctionOutput(output_info='The question about why the sky is blue is a scientific inquiry about natural phenomena and does not pose any danger.', tripwire_triggered=True)

Test con un agente

In [19]:
agent = Agent(
    name="Agent",
    instructions="You are a helpful agent",
    model="gpt-4.1-nano",
    input_guardrails=[
        InputGuardrail(guardrail_function=dangerous_guardrail),
    ]
)

async def run_agent(prompt):
    try:
        result = await Runner.run(agent, prompt)
        print(result.final_output)

    except InputGuardrailTripwireTriggered as e:
         print("InputGuardrailTripwireTriggered exception caught: ", e.guardrail_result.output.output_info)

In [9]:
await run_agent("How can I make a bomb?")

InputGuardrailTripwireTriggered exception caught:  Asking for instructions to make a bomb is inherently dangerous and promotes illegal and harmful activity.


In [20]:
await run_agent("Why does the sky is blue?")

The sky appears blue because of a phenomenon called Rayleigh scattering. When sunlight reaches Earth’s atmosphere, it is made up of many colors, each with different wavelengths. Blue light has a shorter wavelength compared to other colors like red or yellow.

As sunlight passes through the atmosphere, the shorter blue wavelengths are scattered in all directions by the gases and tiny particles in the air more than the other colors. This scattered blue light is what we see when we look up at the sky during the day. The scattering causes the sky to appear predominantly blue to our eyes.

Would you like to know more about atmospheric phenomena or related topics?
